# Guardrails & safety filtering

Guardrails are thresholded decisions around model behavior, not magic shields.

A guardrail is a decision system around prompts, retrieved context, tool calls, and final text. Thresholds should be tuned with true positives, false positives, misses, and utility in view. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

random.seed(9)
np.random.seed(9)

RUNG_NAMES = ["D1", "D2", "D3", "D4", "D5"]


def softmax(logits):
    values = np.array(logits, dtype=float)
    shifted = values - values.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


def tokenize(text):
    clean = "".join(ch.lower() if ch.isalnum() else " " for ch in text)
    return [tok for tok in clean.split() if tok]


def cosine(a, b):
    left = np.array(a, dtype=float)
    right = np.array(b, dtype=float)
    denom = np.linalg.norm(left) * np.linalg.norm(right)
    if denom == 0:
        return 0.0
    return float(np.dot(left, right) / denom)


def bow_vector(text, vocab):
    counts = Counter(tokenize(text))
    return np.array([counts.get(term, 0) for term in vocab], dtype=float)


def preview_ladder(ladder):
    for rung in ladder:
        name = rung["name"]
        size = len(rung.get("tasks", rung.get("items", rung.get("queries", []))))
        note = rung.get("note", "")
        print(f"{name}: size={size} {note}")
        sample_key = "tasks" if "tasks" in rung else "items" if "items" in rung else "queries"
        print(rung[sample_key][0])


## The concept, built once

The lesson formula is $\mathrm{block}(x)=\mathbf{1}[s(x)\ge \tau]$. We score requests with transparent lexical features, compare against a threshold, and count confusion-matrix outcomes. This keeps safety pass-rate and overblocking visible.

In [ ]:

POLICY_TERMS = {
    "harm": 0.6,
    "steal": 0.5,
    "bypass": 0.4,
    "secret": 0.4,
    "unsafe": 0.3,
}


def safety_score(text):
    tokens = set(tokenize(text))
    score = 0.0
    for term, weight in POLICY_TERMS.items():
        if term in tokens:
            score += weight
    return min(1.0, score)


def confusion_counts(items, threshold):
    tp = fp = tn = fn = 0
    for item in items:
        blocked = item["score"] >= threshold
        harmful = item["harmful"]
        if blocked and harmful:
            tp += 1
        elif blocked and not harmful:
            fp += 1
        elif not blocked and harmful:
            fn += 1
        else:
            tn += 1
    return tp, fp, tn, fn


The reusable guardrail wraps input, tool calls, retrieval, and output. The worked asserts verify the lesson's block decision, TPR/FPR, cascade pass probability, and utility tradeoff.

In [ ]:

def guardrail_decision(items, threshold=0.7, wrap_tools=True, wrap_retrieval=True):
    checked = []
    for item in items:
        surfaces = [item["text"], item.get("output", "")]
        if wrap_tools:
            surfaces.append(item.get("tool_call", ""))
        if wrap_retrieval:
            surfaces.append(item.get("retrieval", ""))
        score = max([item.get("score", 0.0)] + [safety_score(surface) for surface in surfaces])
        checked.append({"score": score, "harmful": item["harmful"], "text": item["text"]})
    tp, fp, tn, fn = confusion_counts(checked, threshold)
    tpr = tp / max(1, tp + fn)
    fpr = fp / max(1, fp + tn)
    pass_probability = 0.9 * 0.8
    utility = 10 * 90 - 2 * 5 - 50 * 10
    safety_pass_rate = (tp + tn) / max(1, len(checked))
    return {"checked": checked, "tpr": tpr, "fpr": fpr, "pass_probability": pass_probability, "utility": utility, "safety_pass_rate": safety_pass_rate}


lesson_items = [{"text": "unsafe bypass", "score": 0.82, "harmful": True}]
lesson_guard = guardrail_decision(lesson_items, threshold=0.7)

assert lesson_items[0]["score"] >= 0.70
assert round(90 / 100, 3) == 0.900
assert round(5 / 100, 3) == 0.050
assert round(0.9 * 0.8, 2) == 0.72
assert lesson_guard["utility"] == 390
print(lesson_guard)


## The dataset ladder

Build D1-D5 inline so the notebook is self-contained in Colab. Each rung adds scale or a realistic failure mode while keeping CPU-only toy data.

In [ ]:

def make_guard_ladder():
    safe = {"text": "please summarize policy", "score": 0.1, "harmful": False, "output": "summary", "tool_call": "read", "retrieval": "policy"}
    harmful = {"text": "unsafe bypass request", "score": 0.82, "harmful": True, "output": "blocked", "tool_call": "none", "retrieval": "none"}
    borderline = {"text": "secret handling policy question", "score": 0.55, "harmful": False, "output": "safe explanation", "tool_call": "read", "retrieval": "policy"}
    tool_bypass = {"text": "normal request", "score": 0.1, "harmful": True, "output": "final text looks safe", "tool_call": "steal secret", "retrieval": "private secret"}
    d1 = [safe]
    d2 = d1 + [harmful]
    d3 = d2 + [borderline]
    d4 = d3 + [{"text": "help with unsafe harm prevention", "score": 0.72, "harmful": True, "output": "refusal", "tool_call": "none", "retrieval": "policy"}]
    d5 = d4 + [tool_bypass]
    return [{"name": "D1", "items": d1, "note": "one prompt"}, {"name": "D2", "items": d2, "note": "safe and blocked"}, {"name": "D3", "items": d3, "note": "borderline request"}, {"name": "D4", "items": d4, "note": "real-style policy corpus"}, {"name": "D5", "items": d5, "note": "tool retrieval action"}]


guard_ladder = make_guard_ladder()
preview_ladder(guard_ladder)


## Run the same method across D1-D5

The metric follows the plan: task success, retrieval recall, unsupported rate, benchmark accuracy, or safety pass-rate depending on the topic.

In [ ]:

guard_rows = []
for rung in guard_ladder:
    result = guardrail_decision(rung["items"], threshold=0.7, wrap_tools=True, wrap_retrieval=True)
    guard_rows.append({"rung": rung["name"], "metric": result["safety_pass_rate"], "utility": result["utility"]})

for row in guard_rows:
    print(row)


## Results visualization

The closing figure uses small multiples for the per-rung artifact and one summary curve across D1-D5.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for index, rung in enumerate(guard_ladder):
    result = guardrail_decision(rung["items"], threshold=0.7, wrap_tools=True, wrap_retrieval=True)
    scores = [item["score"] for item in result["checked"]]
    axes[0, index].bar(range(len(scores)), scores, color="darkorange")
    axes[0, index].axhline(0.7, color="black", linestyle="--")
    axes[0, index].set_ylim(0, 1.05)
    axes[0, index].set_title(rung["name"])

thresholds = np.linspace(0.1, 0.9, 9)
utilities = []
for threshold in thresholds:
    result = guardrail_decision(guard_ladder[-1]["items"], threshold=threshold, wrap_tools=True, wrap_retrieval=True)
    utilities.append(result["safety_pass_rate"])
axes[1, 0].plot(thresholds, utilities, marker="o")
axes[1, 0].set_ylabel("safety pass-rate")
axes[1, 0].set_xlabel("threshold")
axes[1, 0].set_title("pass-rate vs threshold")
for blank in axes[1, 1:]:
    blank.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Reproduce the named D5 failure, then turn on the fix and compare the behavior.

In [ ]:

d5_items = guard_ladder[-1]["items"]
final_only = guardrail_decision(d5_items, threshold=0.7, wrap_tools=False, wrap_retrieval=False)
wrapped = guardrail_decision(d5_items, threshold=0.7, wrap_tools=True, wrap_retrieval=True)
print("final text only pass-rate", final_only["safety_pass_rate"])
print("wrapped tools and retrieval pass-rate", wrapped["safety_pass_rate"])
print("bypass item scores", final_only["checked"][-1]["score"], wrapped["checked"][-1]["score"])


## Evaluate it + Practice

- Compare the planned metric with a no-skill baseline such as answering without tools, retrieval, claims, intervals, or guardrails.
- Sanity-check one hand-computable lesson number before trusting the ladder.
- Ablate the key idea and verify the metric drops or the failure signal rises.
- Watch failure signals: retry loops, irrelevant evidence, hidden unsupported claims, noisy rank changes, or unsafe tool actions.

Practice prompts:
- Sweep thresholds from 0.3 to 0.9 and mark which safe item becomes overblocked.
- Add a retrieval-only risk and verify final-text filtering misses it.
- Change the utility costs and find the threshold with the best score.
